# LLM Asset Enrichment for Vertex and the Organizational Catalog

Run this notebook in **ArcGIS Pro 3.7.2**. It enriches the separate Vertex and six-facility outputs while preserving deterministic asset facts and never modifying `Vertex_Assets.gdb`.

Ollama is a separate Windows application. ArcGIS Pro Deep Learning Essentials provides libraries such as PyTorch, Transformers, and the ArcGIS API for Python, but it does **not** install the Ollama executable, Ollama service, or model files.

## 1. Inspect the ArcGIS Pro Python Environment

In [2]:
from datetime import date
import importlib.metadata
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import arcpy
import pandas as pd

REPOSITORY = Path(r"c:\dev\sandbox\demo-data-generator")
SOURCE_GDB = Path(
    r"c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\Vertex_Assets.gdb"
)
VERTEX_OUTPUT = SOURCE_GDB.with_name("Vertex_Assets_Enriched.gdb")
CATALOG_OUTPUT = SOURCE_GDB.with_name("Organizational_Asset_Catalog.gdb")
RUN_VERTEX = False
RUN_CATALOG = True
ALLOW_OVERWRITE = True

LLM_ENDPOINT = "http://localhost:11434/api/generate"
LLM_TAGS_ENDPOINT = "http://localhost:11434/api/tags"
LLM_MODEL = "mistral"
VERTEX_LLM_MAX_REQUESTS = 250
CATALOG_LLM_MAX_REQUESTS = 500
LLM_TIMEOUT_SECONDS = 120

sys.path.insert(0, str(REPOSITORY / "src"))
from indoors_demo_generator.narratives import LocalJsonNarrator, narrative_run_metadata

package_names = (
    "arcgis",
    "torch",
    "transformers",
    "accelerate",
    "bitsandbytes",
    "safetensors",
)
package_rows = []
for package_name in package_names:
    try:
        package_version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        package_version = "Not installed"
    package_rows.append({"package": package_name, "version": package_version})

environment = {
    "arcgis_pro_version": arcpy.GetInstallInfo().get("Version"),
    "python_environment": os.environ.get("CONDA_DEFAULT_ENV", "Unknown"),
    "python_executable": sys.executable,
    "source_geodatabase": str(SOURCE_GDB),
    "vertex_output": str(VERTEX_OUTPUT),
    "catalog_output": str(CATALOG_OUTPUT),
}
print(json.dumps(environment, indent=2))
display(pd.DataFrame(package_rows))

{
  "arcgis_pro_version": "3.7.2",
  "python_environment": "Unknown",
  "python_executable": "C:\\Program Files\\ArcGIS\\Pro\\bin\\ArcGISPro.exe",
  "source_geodatabase": "c:\\Workspace\\Sandbox\\data\\ILM_asset_gen\\Medical Indoors_3141f0\\GDBs\\Vertex_Assets.gdb",
  "vertex_output": "c:\\Workspace\\Sandbox\\data\\ILM_asset_gen\\Medical Indoors_3141f0\\GDBs\\Vertex_Assets_Enriched.gdb",
  "catalog_output": "c:\\Workspace\\Sandbox\\data\\ILM_asset_gen\\Medical Indoors_3141f0\\GDBs\\Organizational_Asset_Catalog.gdb"
}


,package,version
0,arcgis,2.4.3
1,torch,2.9.1
2,transformers,4.51.3
3,accelerate,1.13.0
4,bitsandbytes,0.49.1
5,safetensors,0.7.0


## 2. Check Whether Ollama Is Installed

Deep Learning Essentials does not install Ollama. This check looks for the separate `ollama.exe` application and reports its version without changing the ArcGIS Pro environment.

In [3]:
ollama_install_path = (
    Path(os.environ.get("LOCALAPPDATA", "")) / "Programs" / "Ollama" / "ollama.exe"
)
OLLAMA_EXECUTABLE = shutil.which("ollama")
if OLLAMA_EXECUTABLE is None and ollama_install_path.is_file():
    OLLAMA_EXECUTABLE = str(ollama_install_path)

ollama_version = None
ollama_install_error = None
if OLLAMA_EXECUTABLE:
    try:
        result = subprocess.run(
            [OLLAMA_EXECUTABLE, "--version"],
            capture_output=True,
            text=True,
            timeout=15,
            check=True,
        )
        ollama_version = (result.stdout or result.stderr).strip()
    except (OSError, subprocess.SubprocessError) as error:
        ollama_install_error = f"{type(error).__name__}: {error}"
else:
    ollama_install_error = (
        "Ollama was not found on PATH or at its standard per-user Windows path. "
        "Install it from https://ollama.com/download/windows, then restart ArcGIS Pro."
    )

print(json.dumps({
    "installed": OLLAMA_EXECUTABLE is not None,
    "executable": OLLAMA_EXECUTABLE,
    "version": ollama_version,
    "error": ollama_install_error,
}, indent=2))

{
  "installed": true,
  "executable": "C:\\Users\\jon12828\\AppData\\Local\\Programs\\Ollama\\ollama.EXE",
  "version": "ollama version is 0.35.0",
  "error": null
}


## 3. Verify the Ollama Service

The executable and service are separate checks. Ollama may be installed while its localhost API is stopped. This cell queries the read-only model-tags endpoint.

In [4]:
ollama_tags = None
ollama_service_error = None
try:
    with urlopen(LLM_TAGS_ENDPOINT, timeout=10) as response:
        ollama_tags = json.loads(response.read().decode("utf-8"))
    if not isinstance(ollama_tags, dict) or not isinstance(ollama_tags.get("models"), list):
        raise ValueError("Ollama returned an invalid model-list response.")
except (OSError, TimeoutError, URLError, HTTPError, ValueError, json.JSONDecodeError) as error:
    ollama_service_error = f"{type(error).__name__}: {error}"

OLLAMA_SERVICE_AVAILABLE = ollama_tags is not None
print(json.dumps({
    "service_available": OLLAMA_SERVICE_AVAILABLE,
    "endpoint": LLM_TAGS_ENDPOINT,
    "error": ollama_service_error,
}, indent=2))

{
  "service_available": true,
  "endpoint": "http://localhost:11434/api/tags",
  "error": null
}


## 4. List Available Ollama Models

The configured model must already be downloaded by Ollama. Model weights are not supplied by ArcGIS Pro or this repository.

In [5]:
model_rows = []
if ollama_tags:
    for model in ollama_tags["models"]:
        model_rows.append({
            "name": model.get("name"),
            "size_gb": round(model.get("size", 0) / 1_000_000_000, 2),
            "modified_at": model.get("modified_at"),
            "family": model.get("details", {}).get("family"),
            "parameter_size": model.get("details", {}).get("parameter_size"),
            "quantization": model.get("details", {}).get("quantization_level"),
        })
models = pd.DataFrame(model_rows)
display(models)

available_model_names = set(models["name"]) if not models.empty else set()
MODEL_AVAILABLE = LLM_MODEL in available_model_names or any(
    name.split(":", 1)[0] == LLM_MODEL for name in available_model_names
)
print(f"Configured model available: {MODEL_AVAILABLE}")

,name,size_gb,modified_at,family,parameter_size,quantization
0,mistral:latest,4.37,2026-09-29T15:26:24.0113252-04:00,llama,7.2B,Q4_K_M


Configured model available: True


## 5. Send a Prompt to Ollama

The generic helper below is useful for diagnosing Ollama itself. The project then performs a stricter probe through `LocalJsonNarrator`, which accepts only `{"text": "..."}` and uses deterministic fallback text on malformed output.

In [6]:
def ollama_generate(prompt, model=LLM_MODEL, timeout_seconds=LLM_TIMEOUT_SECONDS):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
    }
    request = Request(
        LLM_ENDPOINT,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=timeout_seconds) as response:
            result = json.loads(response.read().decode("utf-8"))
    except (OSError, TimeoutError, URLError, HTTPError) as error:
        raise RuntimeError(f"Ollama request failed: {error}") from error
    except json.JSONDecodeError as error:
        raise RuntimeError("Ollama returned invalid JSON.") from error
    generated = result.get("response")
    if not isinstance(generated, str) or not generated.strip():
        raise RuntimeError("Ollama returned an empty or invalid response.")
    return generated.strip()

probe_narrator = LocalJsonNarrator(
    LLM_ENDPOINT,
    LLM_MODEL,
    timeout_seconds=LLM_TIMEOUT_SECONDS,
    max_requests=1,
)
probe_text = probe_narrator.narrate(
    "inspection",
    {
        "asset_type": "Air Handling Unit",
        "asset_class": "Building Services",
        "condition": "Maintenance Due",
        "criticality": "High",
        "lifecycle_stage": "Within Service Life",
    },
    "Deterministic probe fallback.",
)
probe_metadata = narrative_run_metadata(probe_narrator)
LLM_READY = probe_metadata["statistics"]["accepted_responses"] == 1
print(probe_text)
print(json.dumps(probe_metadata, indent=2))
if not LLM_READY:
    print(
        "LLM preflight failed. Ensure the Ollama Windows application is running, "
        "the configured model is installed, and the localhost API is available."
    )

The Air Handling Unit, classified as Building Services, requires maintenance. Given its high criticality and current lifecycle stage being within its service life, immediate attention is advised.
{
  "enabled": true,
  "endpoint": "http://localhost:11434/api/generate",
  "model": "mistral",
  "maximum_requests": 1,
  "fallback": "deterministic",
  "statistics": {
    "total_calls": 1,
    "requests_sent": 1,
    "accepted_responses": 1,
    "fallback_responses": 0,
    "budget_fallbacks": 0,
    "invalid_response_fallbacks": 0,
    "request_error_fallbacks": 0
  }
}


## 6. Process ArcGIS Data with Ollama

Only non-sensitive synthetic asset facts are sent to localhost. Vertex offers inspection, repair, and work-order narratives to the model. The scaled catalog offers latest inspections, repairs, and work orders; older inspections stay deterministic. Request budgets bound runtime, and every rejected or skipped response uses deterministic fallback text.

In [7]:
importlib.invalidate_caches()
for module_name in tuple(sys.modules):
    if module_name == "indoors_demo_generator" or module_name.startswith(
        "indoors_demo_generator."
    ):
        del sys.modules[module_name]

from indoors_demo_generator.narratives import LocalJsonNarrator, narrative_run_metadata

print("Project modules refreshed from current source files.")

Project modules refreshed from current source files.


In [8]:
def load_script(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load project script: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

vertex_enrichment = load_script(
    "vertex_enrichment",
    REPOSITORY / "scripts" / "enrich_vertex_asset_history.py",
)
catalog_builder = load_script(
    "catalog_builder",
    REPOSITORY / "scripts" / "build_organizational_asset_catalog.py",
)
print("Project workflows loaded.")

Project workflows loaded.


In [9]:
def run_vertex():
    if not RUN_VERTEX:
        print("Vertex enrichment skipped.")
        return None
    output = VERTEX_OUTPUT.resolve()
    manifest_path = vertex_enrichment._manifest_path(output)
    if output.exists() or arcpy.Exists(str(output)):
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                f"Vertex output exists: {output}. Set ALLOW_OVERWRITE=True to replace it."
            )
        vertex_enrichment._delete_output(output)
    manifest_path.unlink(missing_ok=True)

    config = vertex_enrichment.EnrichmentConfig(
        reference_date=date(2026, 9, 28),
        inspection_start=date(2021, 9, 28),
        inspection_end=date(2026, 3, 28),
        seed=3141,
    )
    source_schema = vertex_enrichment._schema(str(SOURCE_GDB))
    vertex_enrichment._ensure_empty_source_children(source_schema)
    narrator = LocalJsonNarrator(
        LLM_ENDPOINT,
        LLM_MODEL,
        timeout_seconds=LLM_TIMEOUT_SECONDS,
        max_requests=VERTEX_LLM_MAX_REQUESTS,
    )
    try:
        vertex_enrichment._create_minimal_output(str(SOURCE_GDB), str(output))
        output_source = vertex_enrichment.validate_indoors_source(arcpy, str(output))
        output_schema = vertex_enrichment.create_asset_management_schema(
            arcpy, str(output), output_source
        )
        for field_name, field_type, length in vertex_enrichment.LIFECYCLE_EXTENSION_FIELDS:
            vertex_enrichment._add_field(
                output_schema.asset_lifecycle, field_name, field_type, length
            )
        vertex_enrichment._copy_asset_schema_rows(
            source_schema, output_schema, str(output)
        )
        assets = vertex_enrichment._asset_seeds(output_schema)
        _, facility_names = vertex_enrichment._hierarchy(str(output))
        plan = vertex_enrichment.build_enrichment_plan(
            assets,
            config,
            narrator=narrator,
            facility_name=next(iter(facility_names.values())),
        )
        vertex_enrichment._apply_plan(output_schema, plan)
        validation = vertex_enrichment._validate_output(
            str(SOURCE_GDB), str(output), output_schema, config
        )
        manifest = {
            "created_utc": vertex_enrichment.DateTime.now(
                vertex_enrichment.timezone.utc
            ).isoformat(timespec="seconds"),
            "source_geodatabase": str(SOURCE_GDB),
            "output_geodatabase": str(output),
            "source_modified": False,
            "reference_date": config.reference_date.isoformat(),
            "inspection_window": {
                "start": config.inspection_start.isoformat(),
                "end": config.inspection_end.isoformat(),
            },
            "seed": config.seed,
            "narrative_generation": narrative_run_metadata(narrator),
            "validation": validation,
        }
        manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        arcpy.management.ClearWorkspaceCache()
        print(f"Vertex output: {output}")
        return manifest
    except Exception:
        vertex_enrichment._delete_output(output)
        manifest_path.unlink(missing_ok=True)
        raise

In [10]:
def run_catalog():
    if not RUN_CATALOG:
        print("Organizational catalog skipped.")
        return None
    if catalog_builder.SOURCE_GDB.resolve() != SOURCE_GDB.resolve():
        raise RuntimeError("Catalog builder source does not match notebook SOURCE_GDB.")
    if catalog_builder.VERTEX_ENRICHED_GDB.resolve() != VERTEX_OUTPUT.resolve():
        raise RuntimeError("Catalog builder Vertex dependency does not match notebook VERTEX_OUTPUT.")
    if catalog_builder.OUTPUT_GDB.resolve() != CATALOG_OUTPUT.resolve():
        raise RuntimeError("Catalog builder output does not match notebook CATALOG_OUTPUT.")
    if not arcpy.Exists(str(VERTEX_OUTPUT)):
        raise FileNotFoundError(
            f"Catalog generation requires the reviewed Vertex enrichment output: {VERTEX_OUTPUT}"
        )
    catalog_builder.validate_indoors_source(arcpy, str(VERTEX_OUTPUT))
    vertex_schema = catalog_builder._schema(str(VERTEX_OUTPUT))
    required_vertex_datasets = (
        vertex_schema.indoor_asset_locations,
        vertex_schema.exterior_asset_points,
        vertex_schema.asset_lifecycle,
        vertex_schema.inspections,
        vertex_schema.repairs,
        vertex_schema.work_orders,
    )
    missing_vertex_datasets = [
        path for path in required_vertex_datasets if not arcpy.Exists(path)
    ]
    if missing_vertex_datasets:
        raise RuntimeError(
            "Vertex enrichment output is incomplete: "
            + ", ".join(Path(path).name for path in missing_vertex_datasets)
        )

    output = CATALOG_OUTPUT.resolve()
    original_manifest = catalog_builder.MANIFEST_PATH
    original_log = catalog_builder.LOG_PATH
    build_output = output.with_name(f"{output.stem}_building.gdb")
    backup_output = output.with_name(f"{output.stem}_previous.gdb")
    failed_output = output.with_name(f"{output.stem}_failed_promotion.gdb")
    build_manifest = original_manifest.with_name(f"{original_manifest.stem}_building.json")
    backup_manifest = original_manifest.with_name(f"{original_manifest.stem}_previous.json")
    build_log = original_log.with_name(f"{original_log.stem}_building.log")
    backup_log = original_log.with_name(f"{original_log.stem}_previous.log")

    def workspace_exists(path):
        return path.exists() or arcpy.Exists(str(path))

    def catalog_complete(path):
        schema = catalog_builder._schema(str(path))
        required = (
            path / "Indoors" / "Sites",
            path / "Indoors" / "Facilities",
            path / "Indoors" / "Levels",
            path / "Indoors" / "Units",
            schema.indoor_asset_locations,
            schema.exterior_asset_points,
            schema.asset_lifecycle,
            schema.inspections,
            schema.repairs,
            schema.work_orders,
        )
        return workspace_exists(path) and all(arcpy.Exists(str(item)) for item in required)

    def delete_workspace(path):
        arcpy.management.ClearWorkspaceCache()
        if not workspace_exists(path):
            return
        try:
            arcpy.management.Delete(str(path))
        except arcpy.ExecuteError as error:
            raise RuntimeError(
                f"Unable to remove workspace {path}. Close ArcGIS layers, tables, and Catalog "
                "views that reference it, then retry."
            ) from error
        if workspace_exists(path):
            raise RuntimeError(f"Unable to remove workspace: {path}")

    def rename_workspace(source, target):
        arcpy.management.ClearWorkspaceCache()
        if workspace_exists(target):
            raise RuntimeError(f"Promotion target already exists: {target}")
        try:
            source.rename(target)
        except (OSError, PermissionError) as error:
            raise PermissionError(
                f"Cannot rename {source} to {target}. Close or remove every ArcGIS Pro layer, "
                "standalone table, attribute table, and Catalog view that references the old "
                "catalog, then rerun only the final workflow cell. The validated staged build "
                f"will be reused at {build_output}."
            ) from error

    def load_validated_build():
        manifest = json.loads(build_manifest.read_text(encoding="utf-8"))
        expected_validation = {
            "sites": 6,
            "facilities": 6,
            "levels": 32,
            "units": 1025,
            "indoor_assets": 4852,
            "exterior_assets": 72,
            "lifecycle_assets": 4852,
            "orphan_references": 0,
            "preserved_vertex_assets": 1003,
        }
        validation = manifest.get("validation", {})
        mismatches = {
            name: {"expected": expected, "actual": validation.get(name)}
            for name, expected in expected_validation.items()
            if validation.get(name) != expected
        }
        if mismatches or not catalog_complete(build_output):
            raise RuntimeError(
                "The staged catalog is incomplete or failed its expected validation contract: "
                + json.dumps(mismatches, sort_keys=True)
            )
        return manifest

    build_workspace_exists = workspace_exists(build_output)
    build_manifest_exists = build_manifest.exists()
    if build_workspace_exists != build_manifest_exists:
        raise RuntimeError(
            "Incomplete staged catalog artifacts remain. Preserve them for diagnosis or remove "
            f"both {build_output} and {build_manifest} before rebuilding."
        )

    original_paths = (
        catalog_builder.OUTPUT_GDB,
        catalog_builder.MANIFEST_PATH,
        catalog_builder.LOG_PATH,
    )
    try:
        if build_workspace_exists:
            manifest = load_validated_build()
            print(f"Reusing validated staged catalog: {build_output}")
        else:
            if workspace_exists(output) and not ALLOW_OVERWRITE:
                raise FileExistsError(
                    f"Catalog output exists: {output}. Set ALLOW_OVERWRITE=True to replace it."
                )
            stale_artifacts = [
                path
                for path in (backup_output, failed_output, backup_manifest, backup_log)
                if path.exists() or (path.suffix.lower() == ".gdb" and arcpy.Exists(str(path)))
            ]
            if stale_artifacts:
                raise RuntimeError(
                    "Previous catalog promotion artifacts remain: "
                    + ", ".join(str(path) for path in stale_artifacts)
                )

            narrator = LocalJsonNarrator(
                LLM_ENDPOINT,
                LLM_MODEL,
                timeout_seconds=LLM_TIMEOUT_SECONDS,
                max_requests=CATALOG_LLM_MAX_REQUESTS,
            )
            catalog_builder.OUTPUT_GDB = build_output
            catalog_builder.MANIFEST_PATH = build_manifest
            catalog_builder.LOG_PATH = build_log
            catalog_builder.main(narrator)
            manifest = json.loads(build_manifest.read_text(encoding="utf-8"))
            manifest["output_geodatabase"] = str(output)
            build_manifest.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
            manifest = load_validated_build()

        had_output = workspace_exists(output)
        output_complete = catalog_complete(output)
        backup_complete = catalog_complete(backup_output)
        if workspace_exists(backup_output):
            if output_complete:
                raise RuntimeError(
                    "Both the canonical and backup catalogs are complete; promotion state is "
                    "ambiguous and no files were changed."
                )
            if not backup_complete:
                raise RuntimeError(
                    f"The prior catalog backup is incomplete: {backup_output}. No files were changed."
                )
            if had_output:
                if workspace_exists(failed_output):
                    raise RuntimeError(f"Failed-promotion workspace already exists: {failed_output}")
                rename_workspace(output, failed_output)
        elif had_output:
            if not output_complete:
                raise RuntimeError(
                    f"The canonical catalog is incomplete and no prior backup exists: {output}"
                )
            rename_workspace(output, backup_output)

        if original_manifest.exists() and not backup_manifest.exists():
            original_manifest.rename(backup_manifest)
        if original_log.exists() and not backup_log.exists():
            original_log.rename(backup_log)

        promoted_output = False
        try:
            rename_workspace(build_output, output)
            promoted_output = True
            if not catalog_complete(output):
                raise RuntimeError(f"Promoted catalog is incomplete: {output}")
            build_manifest.rename(original_manifest)
        except Exception:
            if promoted_output and workspace_exists(output) and not workspace_exists(build_output):
                output.rename(build_output)
            if not workspace_exists(output) and workspace_exists(backup_output):
                backup_output.rename(output)
            if not original_manifest.exists() and backup_manifest.exists():
                backup_manifest.rename(original_manifest)
            if not original_log.exists() and backup_log.exists():
                backup_log.rename(original_log)
            raise

        if build_log.exists():
            try:
                build_log.rename(original_log)
            except OSError as error:
                print(f"Catalog promoted; build log promotion deferred: {error}")
        for obsolete_workspace in (backup_output, failed_output):
            try:
                delete_workspace(obsolete_workspace)
            except RuntimeError as error:
                print(f"Catalog promoted; cleanup deferred: {error}")
        backup_manifest.unlink(missing_ok=True)
        backup_log.unlink(missing_ok=True)
        print(f"Organizational catalog: {output}")
        return manifest
    finally:
        catalog_builder.OUTPUT_GDB = original_paths[0]
        catalog_builder.MANIFEST_PATH = original_paths[1]
        catalog_builder.LOG_PATH = original_paths[2]

## 7. Handle Missing Installation or Service Errors

This final cell stops before any geodatabase write unless the source exists and the strict model probe succeeded. Each workflow retains its own schema and cleanup handling. If preflight fails, open the Ollama Windows application, confirm the model is installed, and rerun the notebook from the top. Do not start a second `ollama serve` process when the desktop application is already listening on port `11434`.

In [11]:
def require_ready():
    problems = []
    if not arcpy.Exists(str(SOURCE_GDB)):
        problems.append(f"Source geodatabase was not found: {SOURCE_GDB}")
    if not OLLAMA_SERVICE_AVAILABLE:
        problems.append(
            "Ollama service is unavailable. Open the Ollama Windows application outside ArcGIS Pro."
        )
    if not MODEL_AVAILABLE:
        problems.append(
            f"Model {LLM_MODEL!r} is unavailable. Run `ollama pull {LLM_MODEL}` outside ArcGIS Pro."
        )
    if not LLM_READY:
        problems.append(
            "The model failed the strict JSON response contract; inspect the probe telemetry above."
        )
    if problems:
        raise RuntimeError("\n".join(problems))


def validate_narrative_statistics(manifest):
    statistics = manifest["narrative_generation"]["statistics"]
    expected_total = statistics["accepted_responses"] + statistics["fallback_responses"]
    if statistics["total_calls"] != expected_total:
        raise RuntimeError(
            f"Narrative call accounting is inconsistent: {statistics['total_calls']} total, "
            f"expected {expected_total}."
        )
    expected_fallbacks = (
        statistics["budget_fallbacks"]
        + statistics["invalid_response_fallbacks"]
        + statistics["request_error_fallbacks"]
    )
    if statistics["fallback_responses"] != expected_fallbacks:
        raise RuntimeError(
            f"Narrative fallback accounting is inconsistent: "
            f"{statistics['fallback_responses']} fallbacks, expected {expected_fallbacks}."
        )


require_ready()
results = {}
failures = {}
for workflow_name, workflow in (("vertex", run_vertex), ("catalog", run_catalog)):
    try:
        manifest = workflow()
        if manifest is not None:
            validate_narrative_statistics(manifest)
            results[workflow_name] = manifest["narrative_generation"]
    except (FileExistsError, RuntimeError, ValueError, OSError, arcpy.ExecuteError) as error:
        failures[workflow_name] = f"{type(error).__name__}: {error}"
        print(f"{workflow_name} failed: {failures[workflow_name]}")

print("Narrative telemetry:")
print(json.dumps(results, indent=2))
if failures:
    raise RuntimeError("One or more workflows failed:\n" + json.dumps(failures, indent=2))
print("LLM asset enrichment completed.")

Vertex enrichment skipped.
Reusing validated staged catalog: C:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\Organizational_Asset_Catalog_building.gdb
Organizational catalog: C:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\Organizational_Asset_Catalog.gdb
Narrative telemetry:
{
  "catalog": {
    "enabled": true,
    "endpoint": "http://localhost:11434/api/generate",
    "model": "mistral",
    "maximum_requests": 500,
    "fallback": "deterministic",
    "statistics": {
      "total_calls": 5912,
      "requests_sent": 500,
      "accepted_responses": 498,
      "fallback_responses": 5414,
      "budget_fallbacks": 5412,
      "invalid_response_fallbacks": 0,
      "request_error_fallbacks": 2
    }
  }
}
LLM asset enrichment completed.
